[Reference](https://medium.com/@pankaj_pandey/6de02b08b64f)

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np


class ToolSelector:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.tools = []
        self.embeddings = None
    def index(self, tools):
        self.tools = tools
        descriptions = [
            f"{t['name']}: {t.get('description', '')}"
            for t in tools
        ]
        self.embeddings = self.model.encode(descriptions, normalize_embeddings=True)
    def select(self, query, top_k=5):
        query_embedding = self.model.encode([query], normalize_embeddings=True)
        scores = np.dot(self.embeddings, query_embedding.T).flatten()
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [self.tools[i] for i in top_indices if scores[i] > 0.3]

# OpenAI Agents SDK

In [2]:
from pathlib import Path
from agents import Agent, Runner
from agents.mcp import MCPServerStdio


samples_dir = Path(__file__).parent / "sample_files"
async with MCPServerStdio(
    name="Filesystem Server",
    params={
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(samples_dir)],
    },
    cache_tools_list=True,
) as server:
    agent = Agent(
        name="Assistant",
        instructions="Use the files in the sample directory to answer questions.",
        mcp_servers=[server],
    )
    result = await Runner.run(agent, "List the files available to you.")
    print(result.final_output)

In [3]:
from agents.mcp import MCPServerStdio, create_static_tool_filter


filesystem_server = MCPServerStdio(
    params={
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(samples_dir)],
    },
    tool_filter=create_static_tool_filter(allowed_tool_names=["read_file", "write_file"]),
)

# LangGraph

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent


client = MultiServerMCPClient({
    "math": {
        "command": "python",
        "args": ["/path/to/math_server.py"],
        "transport": "stdio",
    },
    "weather": {
        "url": "http://localhost:8000/mcp",
        "transport": "streamable_http",
    }
})
tools = await client.get_tools()
agent = create_react_agent("openai:gpt-4.1", tools)
response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})

# CrewAI

In [5]:
from crewai import Agent, Crew, Task
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters

server_params = StdioServerParameters(
    command="uvx",
    args=["--quiet", "pubmedmcp@0.1.3"],
    env={"UV_PYTHON": "3.12", **os.environ},
)
with MCPServerAdapter(server_params) as tools:
    agent = Agent(
        role="Research Agent",
        goal="Find relevant studies",
        tools=tools,
    )
    task = Task(
        description="Find studies about the topic",
        agent=agent,
        expected_output="A list of relevant studies",
    )
    crew = Crew(agents=[agent], tasks=[task])
    crew.kickoff()

In [6]:
server_params = {"url": "http://localhost:8000/sse"}
with MCPServerAdapter(server_params) as tools:
    # tools is now a list of CrewAI Tools
    agent = Agent(role="Assistant", tools=tools)

# PydanticAI

In [7]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPServerStdio

server = MCPServerStdio('python', args=['mcp_server.py'], timeout=10)
agent = Agent('openai:gpt-4o', toolsets=[server])
async def main():
    result = await agent.run('What is the weather in Paris?')
    print(result.output)

In [8]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPServerStreamableHTTP

server = MCPServerStreamableHTTP('http://localhost:8000/mcp')
agent = Agent('anthropic:claude-sonnet-4-0', toolsets=[server])
async def main():
    result = await agent.run('What is 7 plus 5?')
    print(result.output)

# Agno

In [9]:
from agno.agent import Agent
from agno.models.anthropic import Claude
from agno.tools.mcp import MCPTools

async with MCPTools(
    transport="streamable-http",
    url="https://api.example.com/mcp"
) as mcp_tools:
    agent = Agent(
        model=Claude(id="claude-sonnet-4-5"),
        tools=[mcp_tools],
        markdown=True,
    )
    await agent.aprint_response("What's available?", stream=True)

In [10]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.mcp import MCPTools

async with MCPTools(command="npx -y @modelcontextprotocol/server-filesystem /tmp") as mcp_tools:
    agent = Agent(
        model=OpenAIChat(id="gpt-4o"),
        tools=[mcp_tools],
    )
    await agent.aprint_response("List files", stream=True)

# PraisonAI

In [11]:
from praisonaiagents import Agent, MCP

agent = Agent(
    instructions="You help book apartments on Airbnb.",
    llm="gpt-4o-mini",
    tools=MCP("npx -y @openbnb/mcp-server-airbnb --ignore-robots-txt")
)
agent.start("Search for apartments in Paris for 2 nights")

In [12]:
from praisonaiagents import Agent, MCP


agent = Agent(
    instructions="You can interact with GitHub.",
    llm="gpt-4o-mini",
    tools=MCP("npx -y @modelcontextprotocol/server-github", env={
        "GITHUB_PERSONAL_ACCESS_TOKEN": os.getenv("GITHUB_TOKEN")
    })
)
agent.start("List my repositories")

In [13]:
from praisonaiagents import Agent, MCP


agent = Agent(
    instructions="You can interact with GitHub.",
    llm="gpt-4o-mini",
    tools=MCP("npx -y @modelcontextprotocol/server-github", env={
        "GITHUB_PERSONAL_ACCESS_TOKEN": os.getenv("GITHUB_TOKEN")
    })
)
agent.start("List my repositories")